# Project Name:
# Tata Steel Machine Failure Prediction


# Project Summary:
This capstone project focuses on developing a machine learning solution for predicting machine failure in Tata Steel’s manufacturing environment. The objective is to identify whether a machine is likely to fail based on operational parameters such as air temperature, process temperature, rotational speed, torque, and tool wear.

Predicting machine failure is a critical industrial requirement because unexpected breakdowns can result in production losses, increased maintenance costs, and reduced operational efficiency. A predictive approach allows maintenance teams to take timely action and helps shift the process from reactive maintenance to a more proactive strategy.

The dataset used in this project contains machine operating conditions along with failure indicators. Since failure events are relatively rare compared to normal operating cases, this problem is treated as an imbalanced binary classification task. To address this, the notebook includes data preprocessing, exploratory analysis, feature engineering, model development, and performance evaluation.

GitHub Link:
https://github.com/svenkatesh345/AI_Enginnering/tree/main/TataSteelMachineFailurePrediction

### Problem Statement
The goal of this capstone project is to build a machine learning model that can predict whether a machine will fail based on operational parameters such as temperature, rotational speed, torque, tool wear, and failure-related indicators.

In an industrial setting, early prediction of machine failure is valuable because it helps reduce unplanned downtime, supports preventive maintenance, and improves operational efficiency. Since failure cases are relatively rare compared to non-failure cases, this problem is also an imbalanced binary classification task.

In this notebook, we will perform data understanding, preprocessing, feature engineering, class imbalance handling, model building, threshold tuning, and error analysis to identify a practical and interpretable failure detection solution.

In [ ]:
# SECTION 1: DATA INGESTION AND EXPLORATION

import pandas as pd

# Loading the training dataset from Google Drive.
# This dataset contains machine telemetry data and a 'Machine failure' label,
# which indicates whether a machine failed (1) or not (0).
df = pd.read_csv("/content/drive/MyDrive/Almabetter/Capstone/ML/train.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [ ]:
# Display the shape of the DataFrame (number of rows, number of columns).
# This helps in understanding the size of our dataset.
df.shape

(136429, 14)

In [ ]:
# Display the first 5 rows of the DataFrame.
# This provides a quick overview of the data structure, column names, and sample values.
df.head()

,id,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,0,L50096,L,300.6,309.6,1596,36.1,140,0,0,0,0,0,0
1,1,M20343,M,302.6,312.1,1759,29.1,200,0,0,0,0,0,0
2,2,L49454,L,299.3,308.5,1805,26.5,25,0,0,0,0,0,0
3,3,L53355,L,301.0,310.9,1524,44.3,197,0,0,0,0,0,0
4,4,M24050,M,298.0,309.0,1641,35.4,34,0,0,0,0,0,0


In [ ]:
# Display all column names in the DataFrame.
# This is useful for checking for typos, understanding the features, and planning data preprocessing.
df.columns

Index(['id', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='object')

In [ ]:
# Get a concise summary of the DataFrame.
# This includes the data types of each column, non-null values count, and memory usage.
# It helps in identifying columns with missing values and understanding data types.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136429 entries, 0 to 136428
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       136429 non-null  int64  
 1   Product ID               136429 non-null  object 
 2   Type                     136429 non-null  object 
 3   Air temperature [K]      136429 non-null  float64
 4   Process temperature [K]  136429 non-null  float64
 5   Rotational speed [rpm]   136429 non-null  int64  
 6   Torque [Nm]              136429 non-null  float64
 7   Tool wear [min]          136429 non-null  int64  
 8   Machine failure          136429 non-null  int64  
 9   TWF                      136429 non-null  int64  
 10  HDF                      136429 non-null  int64  
 11  PWF                      136429 non-null  int64  
 12  OSF                      136429 non-null  int64  
 13  RNF                      136429 non-null  int64  
dtypes: f

In [ ]:
# Check for missing values in each column.
# .isnull() returns a boolean DataFrame indicating missing values (True for NaN).
# .sum() then counts the number of True values (missing values) for each column.
df.isnull().sum()

,0
id,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


In [ ]:
# Separate features (X) from the target variable (y).
# 'id' and 'Product ID' are identifiers and not predictive features, so they are dropped.
# 'TWF', 'HDF', 'PWF', 'OSF', 'RNF' are specific failure modes which might be redundant or leak information
# if the goal is to predict 'Machine failure' generally, so they are also dropped.
# 'Machine failure' is the target variable we want to predict.
X = df.drop(['id', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Machine failure'], axis=1)

In [ ]:
# Assign the 'Machine failure' column to 'y', which will be our target variable.
y = df['Machine failure']

In [ ]:
# Display the first 5 rows of the feature DataFrame (X) after dropping certain columns.
# This shows the set of features that will be used for model training.
X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,L,300.6,309.6,1596,36.1,140
1,M,302.6,312.1,1759,29.1,200
2,L,299.3,308.5,1805,26.5,25
3,L,301.0,310.9,1524,44.3,197
4,M,298.0,309.0,1641,35.4,34


In [ ]:
# Count the occurrences of each unique value in the target variable (y).
# This helps in understanding the class distribution and identifying potential class imbalance.
# In this case, it shows a significant imbalance between '0' (no failure) and '1' (failure).
y.value_counts()

,count
Machine failure,
0,134281
1,2148


In [ ]:
# SECTION 2: DATA SPLITTING

from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets.
# X: features, y: target variable.
# test_size=0.2: 20% of the data will be used for testing, 80% for training.
# random_state=42: Ensures reproducibility of the split.
# stratify=y: Maintains the same proportion of target classes in both training and testing sets.
# This is crucial for imbalanced datasets like this one, to ensure both classes are represented in each split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data split into training and testing sets.")

Data split into training and testing sets.


In [ ]:
# Print the shapes of the training and testing feature sets.
# This confirms the number of samples and features in each set.
print(X_train.shape, X_test.shape)

# Print the value counts for the target variable in the training set.
# This verifies that stratification worked and the class distribution is maintained.
print(y_train.value_counts())

# Print the value counts for the target variable in the testing set.
# This also confirms the class distribution in the test set.
print(y_test.value_counts())

(109143, 6) (27286, 6)
Machine failure
0    107425
1      1718
Name: count, dtype: int64
Machine failure
0    26856
1      430
Name: count, dtype: int64


In [ ]:
# Display the data types of the columns in the training feature set.
# This helps in identifying categorical columns that need encoding and numerical columns for scaling.
X_train.dtypes

,0
Type,object
Air temperature [K],float64
Process temperature [K],float64
Rotational speed [rpm],int64
Torque [Nm],float64
Tool wear [min],int64


In [ ]:
# Encoding the Categorical Columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns by selecting columns with 'object' data type from X_train.
cat_cols = X_train.select_dtypes(include='object').columns

# Create a ColumnTransformer to apply OneHotEncoder to categorical columns.
# OneHotEncoder converts categorical variables into a numerical format that ML algorithms can use.
# handle_unknown='ignore': Allows the encoder to ignore categories not seen during training, preventing errors.
# remainder='passthrough': Keeps all other (numerical) columns unchanged.
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')

# Fit the preprocessor on the training data and transform it.
# This learns the categories and applies the encoding.
X_train_enc = preprocessor.fit_transform(X_train)

# Transform the test data using the *fitted* preprocessor.
# It's crucial to use .transform() here, not .fit_transform(), to avoid data leakage from the test set.
X_test_enc = preprocessor.transform(X_test)

In [ ]:
# Balancing the data using SMOTE (Synthetic Minority Over-sampling Technique).
# SMOTE is used to address class imbalance by generating synthetic samples for the minority class.
from imblearn.over_sampling import SMOTE

# Initialize SMOTE with a random_state for reproducibility.
smote = SMOTE(random_state=42)

# Apply SMOTE to the encoded training data.
# This creates new synthetic samples for the minority class ('Machine failure'=1) in X_train_enc and y_train.
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_enc, y_train)

In [ ]:
# Model Building: Initial Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize a RandomForestClassifier model.
# random_state=42 ensures reproducibility of the model's random processes.
model = RandomForestClassifier(random_state=42)

# Train the model using the SMOTE-balanced training data.
# The model learns patterns from the features (X_train_balanced) to predict the target (y_train_balanced).
model.fit(X_train_balanced, y_train_balanced)

# Make predictions on the encoded test set using the trained model.
y_pred = model.predict(X_test_enc)


In [ ]:
# Print the confusion matrix for the initial Random Forest model.
# This matrix shows the counts of true positive, true negative, false positive, and false negative predictions.
print(confusion_matrix(y_test, y_pred))

# Print the classification report, which includes precision, recall, f1-score, and support for each class.
# This provides a more detailed evaluation of the model's performance on both majority and minority classes.
print(classification_report(y_test, y_pred))

[[26488   368]
 [  208   222]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     26856
           1       0.38      0.52      0.44       430

    accuracy                           0.98     27286
   macro avg       0.68      0.75      0.71     27286
weighted avg       0.98      0.98      0.98     27286



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, fbeta_score

# Initialize the Random Forest Classifier with optimized hyperparameters.
# n_estimators=200: Number of trees in the forest.
# max_depth=10: Maximum depth of each tree, controls complexity.
# class_weight='balanced': Automatically adjusts weights inversely proportional to class frequencies.
#                          This helps handle class imbalance directly within the model training.
# random_state=42: Ensures reproducibility.
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

# Fit the model using the SMOTE-balanced training data.
# Training on balanced data helps the model learn patterns for the minority class more effectively.
model.fit(X_train_balanced, y_train_balanced)

# Generate predictions on the encoded test set.
y_pred = model.predict(X_test_enc)

# Evaluate performance using Confusion Matrix and Classification Report.
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Calculate and print the F2 Score (beta=2).
# The F2 score gives twice as much weight to recall as to precision.
# This is important in machine failure prediction where False Negatives (missing a failure) are more costly.
print("\nF2 Score:", fbeta_score(y_test, y_pred, beta=2))

Confusion Matrix:
[[25222  1634]
 [  137   293]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.94      0.97     26856
           1       0.15      0.68      0.25       430

    accuracy                           0.94     27286
   macro avg       0.57      0.81      0.61     27286
weighted avg       0.98      0.94      0.95     27286


F2 Score: 0.4017000274197971


In [ ]:
# Import necessary libraries for numerical operations and evaluation metrics
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score

# Predict probabilities for the positive class (failure) on the encoded test set.
# model.predict_proba returns probabilities for both classes [P(class 0), P(class 1)],
# so we select the probabilities for the positive class (index 1).
# We use X_test because the current model (XGBoost) was trained on the newly engineered and cleaned X_train.
proba = model.predict_proba(X_test)[:, 1]

# Define a range of thresholds to evaluate. This allows us to find an optimal threshold
# for classifying predictions as 'failure' (1) or 'no failure' (0).
# Thresholds are generated from 0.1 to 0.9 (inclusive) with a step of 0.05.
thresholds = np.arange(0.1, 0.91, 0.05)

# Initialize variables to keep track of the best F2 score and the corresponding threshold.
# F2 score is used because false negatives (missing a failure) are typically more costly
# than false positives (false alarm) in machine failure prediction.
best_threshold = 0.5 # Starting with a default threshold
best_f2 = 0 # Starting with a zero F2 score

# Iterate through each threshold to find the one that yields the highest F2 score.
for t in thresholds:
    # Convert predicted probabilities into binary predictions (0 or 1)
    # If the probability is greater than or equal to the current threshold 't', predict 1 (failure);
    # otherwise, predict 0 (no failure).
    y_pred_t = (proba >= t).astype(int)

    # Calculate the F2 score for the current threshold.
    # beta=2 gives twice as much weight to recall as to precision.
    f2 = fbeta_score(y_test, y_pred_t, beta=2)

    # Print the F2 score for the current threshold for monitoring.
    print(f"Threshold: {t:.2f}, F2: {f2:.4f}")

    # Check if the current F2 score is better than the best F2 score found so far.
    if f2 > best_f2:
        best_f2 = f2 # Update the best F2 score
        best_threshold = t # Update the best threshold

# After iterating through all thresholds, print the best threshold and its corresponding F2 score.
print("Best threshold:", best_threshold)
print("Best F2:", best_f2)

# Apply the best found threshold to the predicted probabilities to get the final binary predictions.
y_pred_best = (proba >= best_threshold).astype(int)

# Print the confusion matrix to evaluate the performance of the model at the best threshold.
# The confusion matrix shows True Positives, True Negatives, False Positives, and False Negatives.
print(confusion_matrix(y_test, y_pred_best))

# Print the classification report, which provides precision, recall, f1-score,
# and support for each class, as well as overall averages.
print(classification_report(y_test, y_pred_best))

Threshold: 0.10, F2: 0.1349
Threshold: 0.15, F2: 0.2294
Threshold: 0.20, F2: 0.2748
Threshold: 0.25, F2: 0.3164
Threshold: 0.30, F2: 0.3401
Threshold: 0.35, F2: 0.3607
Threshold: 0.40, F2: 0.3722
Threshold: 0.45, F2: 0.3911
Threshold: 0.50, F2: 0.4017
Threshold: 0.55, F2: 0.4293
Threshold: 0.60, F2: 0.4426
Threshold: 0.65, F2: 0.4597
Threshold: 0.70, F2: 0.4692
Threshold: 0.75, F2: 0.4511
Threshold: 0.80, F2: 0.4180
Threshold: 0.85, F2: 0.3649
Threshold: 0.90, F2: 0.2875
Best threshold: 0.7000000000000002
Best F2: 0.4692367601246106
[[26249   607]
 [  189   241]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     26856
           1       0.28      0.56      0.38       430

    accuracy                           0.97     27286
   macro avg       0.64      0.77      0.68     27286
weighted avg       0.98      0.97      0.98     27286



In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score

# Define a specific set of thresholds to evaluate.
# These might be chosen based on previous iterations or domain knowledge.
thresholds = [0.60, 0.65, 0.70, 0.75, 0.80]

# Initialize a list to store results for each threshold.
results = []

# Loop through each defined threshold.
for t in thresholds:
    # Convert predicted probabilities (proba) into binary predictions (0 or 1).
    # If proba is greater than or equal to the current threshold 't', predict 1 (failure), else 0.
    y_pred_t = (proba >= t).astype(int)

    # Calculate the confusion matrix for the current threshold.
    cm = confusion_matrix(y_test, y_pred_t)
    # Unpack the confusion matrix to get True Negatives (tn), False Positives (fp),
    # False Negatives (fn), and True Positives (tp).
    tn, fp, fn, tp = cm.ravel()

    # Calculate Precision: TP / (TP + FP). Proportion of positive predictions that were correct.
    precision = tp / (tp + fp) if (tp + fp) else 0
    # Calculate Recall: TP / (TP + FN). Proportion of actual positives that were correctly identified.
    recall = tp / (tp + fn) if (tp + fn) else 0
    # Calculate F2 Score: Gives more weight to recall (beta=2).
    f2 = fbeta_score(y_test, y_pred_t, beta=2)

    # Store all calculated metrics for the current threshold in the results list.
    results.append({
        "threshold": t,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": precision,
        "recall": recall,
        "f2": f2
    })

    # Print the threshold, confusion matrix, and classification report for immediate review.
    print(f"\nThreshold: {t}")
    print(cm)
    print(classification_report(y_test, y_pred_t))

# Convert the list of results into a Pandas DataFrame for easier analysis and sorting.
df_results = pd.DataFrame(results)
# Print the DataFrame sorted by F2 score in descending order.
# This helps in quickly identifying the best performing threshold among the tested ones.
print(df_results.sort_values("f2", ascending=False))


Threshold: 0.6
[[25755  1101]
 [  156   274]]
              precision    recall  f1-score   support

           0       0.99      0.96      0.98     26856
           1       0.20      0.64      0.30       430

    accuracy                           0.95     27286
   macro avg       0.60      0.80      0.64     27286
weighted avg       0.98      0.95      0.97     27286


Threshold: 0.65
[[25998   858]
 [  169   261]]
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     26856
           1       0.23      0.61      0.34       430

    accuracy                           0.96     27286
   macro avg       0.61      0.79      0.66     27286
weighted avg       0.98      0.96      0.97     27286


Threshold: 0.7
[[26249   607]
 [  189   241]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     26856
           1       0.28      0.56      0.38       430

    accuracy                           0

In [ ]:
import pandas as pd
import numpy as np

# Apply the best threshold identified from the previous analysis to get final binary predictions.
y_pred_final = (proba >= best_threshold).astype(int)

# Create a copy of the test features. Ensure it's a DataFrame for consistency.
# This handles cases where X_test might be a NumPy array.
data = X_test.copy() if isinstance(X_test, pd.DataFrame) else pd.DataFrame(X_test)

# Add the actual target values, final predictions, and predicted probabilities to the DataFrame.
# This allows for direct comparison and analysis of model performance at a record level.
data["actual"] = y_test.values if hasattr(y_test, "values") else y_test
data["predicted"] = y_pred_final
data["probability"] = proba

# Isolate False Negatives: Actual failure (1) but predicted no failure (0).
# These are critical errors as they represent missed failures.
false_negatives = data[(data["actual"] == 1) & (data["predicted"] == 0)].copy()

# Isolate False Positives: Actual no failure (0) but predicted failure (1).
# These are false alarms.
false_positives = data[(data["actual"] == 0) & (data["predicted"] == 1)].copy()

# Isolate True Positives: Actual failure (1) and predicted failure (1).
true_positives = data[(data["actual"] == 1) & (data["predicted"] == 1)].copy()

# Isolate True Negatives: Actual no failure (0) and predicted no failure (0).
true_negatives = data[(data["actual"] == 0) & (data["predicted"] == 0)].copy()

# Print the counts for each error/correct prediction type.
print("False Negatives:", len(false_negatives))
print("False Positives:", len(false_positives))
print("True Positives:", len(true_positives))
print("True Negatives:", len(true_negatives))

# Save False Negatives and False Positives to CSV files.
# This allows for external analysis of these critical misclassifications.
false_negatives.to_csv("false_negatives.csv", index=False)
false_positives.to_csv("false_positives.csv", index=False)

# Display sample rows for False Negatives and False Positives.
# This gives insight into the characteristics of the data points that the model struggles with.
print("\nSample False Negatives:")
print(false_negatives.head())

print("\nSample False Positives:")
print(false_positives.head())

False Negatives: 189
False Positives: 607
True Positives: 241
True Negatives: 26249

Sample False Negatives:
       Type  Air temperature [K]  Process temperature [K]  \
31786     L                299.7                    310.4   
123450    L                299.8                    311.2   
7442      M                297.4                    308.9   
100577    L                297.4                    308.3   
32990     L                297.4                    308.4   

        Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  actual  \
31786                     1380         49.3               99       1   
123450                    1377         53.6              143       1   
7442                      1449         40.5              204       1   
100577                    1472         46.4              207       1   
32990                     1361         46.2              189       1   

        predicted  probability  
31786           0     0.445820  
123450          0     0.4

In [ ]:
# SECTION 6: ERROR ANALYSIS
# Analyzing the characteristics of False Negatives and False Positives
# to understand where the model struggles and identify patterns.

# Identify numerical columns for statistical analysis.
# Exclude 'actual', 'predicted', and 'probability' columns as they are labels/model outputs.
numeric_cols = data.select_dtypes(include=np.number).columns.drop(["actual", "predicted", "probability"], errors="ignore")

# Calculate mean values for the overall test set, false negatives, and false positives.
# This helps compare feature distributions across different prediction outcomes.
summary = pd.DataFrame({
    "overall_mean": data[numeric_cols].mean(),
    "fn_mean": false_negatives[numeric_cols].mean(),
    "fp_mean": false_positives[numeric_cols].mean()
})

# Calculate the deviation of false negative and false positive means from the overall mean.
# A positive 'fn_diff' means that feature's value is higher in false negatives than overall.
summary["fn_diff"] = summary["fn_mean"] - summary["overall_mean"]
summary["fp_diff"] = summary["fp_mean"] - summary["overall_mean"]

# Sort the summary by 'fn_diff' in descending order.
# This helps identify features whose higher values are most associated with False Negatives (missed failures).
summary = summary.sort_values("fn_diff", ascending=False)

# Save the error analysis summary to a CSV file for further examination.
summary.to_csv("error_analysis_summary.csv")

print("Error Analysis Summary (Feature Deviations):")
print(summary.head(15))

Error Analysis Summary (Feature Deviations):
                         overall_mean      fn_mean      fp_mean    fn_diff  \
Tool wear [min]            104.030932   127.370370   124.675453  23.339439   
Torque [Nm]                 40.371198    44.308995    53.533443   3.937797   
Air temperature [K]        299.865869   300.322751   300.720758   0.456882   
Process temperature [K]    309.942813   310.207407   310.379572   0.264594   
Rotational speed [rpm]    1519.831599  1492.455026  1395.629325 -27.376572   

                            fp_diff  
Tool wear [min]           20.644521  
Torque [Nm]               13.162245  
Air temperature [K]        0.854889  
Process temperature [K]    0.436758  
Rotational speed [rpm]  -124.202274  


In [ ]:
# --- Feature Engineering ---
# We create domain-specific features to capture relationships between operational parameters.
# These ratios and products often help the model identify stress conditions leading to failure more effectively.

import numpy as np
import pandas as pd

# Create a copy of the original feature DataFrame X to add new engineered features.
df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

# 1. Ratio of Torque to Speed: This feature can indicate high load conditions.
#    A high torque at a relatively low rotational speed might suggest the machine is under strain.
#    Adding a small constant (1e-6) to the denominator prevents division by zero.
df["torque_speed_ratio"] = df["Torque [Nm]"] / (df["Rotational speed [rpm]"] + 1e-6)

# 2. Product of Tool Wear and Torque: Represents cumulative stress or work done by the tool.
#    Higher values could indicate a tool nearing its failure point.
df["wear_torque"] = df["Tool wear [min]"] * df["Torque [Nm]"]

# 3. Tool wear relative to speed: This can show how quickly the tool is wearing down per unit of rotation.
#    A high wear rate at a certain speed could be a precursor to failure.
df["wear_speed_ratio"] = df["Tool wear [min]"] / (df["Rotational speed [rpm]"] + 1e-6)

# 4. Torque applied per unit of tool wear: This might indicate the material's resistance or how much torque
#    is being sustained despite increasing wear. A sudden drop could signal an issue.
df["torque_per_wear"] = df["Torque [Nm]"] / (df["Tool wear [min]"] + 1e-6)

print("Engineered Features Preview:")
print(df.head())

Engineered Features Preview:
  Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0    L                300.6                    309.6                    1596   
1    M                302.6                    312.1                    1759   
2    L                299.3                    308.5                    1805   
3    L                301.0                    310.9                    1524   
4    M                298.0                    309.0                    1641   

   Torque [Nm]  Tool wear [min]  torque_speed_ratio  wear_torque  \
0         36.1              140            0.022619       5054.0   
1         29.1              200            0.016543       5820.0   
2         26.5               25            0.014681        662.5   
3         44.3              197            0.029068       8727.1   
4         35.4               34            0.021572       1203.6   

   wear_speed_ratio  torque_per_wear  
0          0.087719         0.257857  
1  

In [ ]:
# SECTION 3: FEATURE ENGINEERING AND PREPROCESSING

import pandas as pd

# Create copies of X_train and X_test to avoid modifying the original DataFrames directly.
X_train = X_train.copy()
X_test = X_test.copy()

# Define a helper function to dynamically get column names.
# This is robust to slight variations in naming (e.g., due to previous sanitization steps).
def get_col_name(df, original):
    sanitized = original.replace('[', '_').replace(']', '_').replace(' ', '_').replace('<', '_').replace('>', '_')
    if original in df.columns:
        return original
    return sanitized

# Apply feature engineering to both training and testing datasets.
for df_split in [X_train, X_test]:
    # Dynamically find the correct column names for 'Torque', 'Rotational speed', and 'Tool wear'.
    torque_col = get_col_name(df_split, "Torque [Nm]")
    speed_col = get_col_name(df_split, "Rotational speed [rpm]")
    wear_col = get_col_name(df_split, "Tool wear [min]")

    # Ratio of Torque to Speed: identifies high-load, low-speed strain. (as explained in previous cell)
    df_split["torque_speed_ratio"] = df_split[torque_col] / (df_split[speed_col] + 1e-6)
    # Wear-Torque product: represents the cumulative work stress on the tool. (as explained in previous cell)
    df_split["wear_torque"] = df_split[wear_col] * df_split[torque_col]
    # Wear relative to speed: tracks efficiency of wear per cycle. (as explained in previous cell)
    df_split["wear_speed_ratio"] = df_split[wear_col] / (df_split[speed_col] + 1e-6)
    # Torque per unit of wear: tracks the resistance of the material. (as explained in previous cell)
    df_split["torque_per_wear"] = df_split[torque_col] / (df_split[wear_col] + 1e-6)

# Encoding categorical 'Type' feature into numerical dummy variables.
# pd.get_dummies converts categorical columns into one-hot encoded format.
# drop_first=True avoids multicollinearity by dropping the first category.
# The 'if' conditions ensure this step is only performed if 'Type' column exists, making the code robust.
if "Type" in X_train.columns:
    X_train = pd.get_dummies(X_train, columns=["Type"], drop_first=True)
if "Type" in X_test.columns:
    X_test = pd.get_dummies(X_test, columns=["Type"], drop_first=True)

# Aligning columns between train and test to ensure consistency after dummy encoding.
# This is crucial because OneHotEncoder might create different numbers of columns if categories differ.
# fill_value=0: Fills new columns (if any exist in train but not in test, or vice-versa) with zeros.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Feature engineering and dummy encoding complete. Handled dynamic column naming.")

Feature engineering and dummy encoding complete. Handled dynamic column naming.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# SECTION 3.1: RE-INITIALIZING TRAINING DATA WITH ROBUST NAMING
# This section re-executes feature engineering and data splitting, ensuring all subsequent model training
# uses the fully engineered and correctly split datasets with robust column naming.

# Create a copy of the original feature DataFrame X.
X = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

# Helper function to get the correct column name, accounting for potential sanitization.
def get_col(df, name):
    sanitized = name.replace('[', '_').replace(']', '_').replace(' ', '_').replace('<', '_').replace('>', '_')
    return name if name in df.columns else sanitized

# Get sanitized column names for feature engineering.
t_col = get_col(X, 'Torque [Nm]')
s_col = get_col(X, 'Rotational speed [rpm]')
w_col = get_col(X, 'Tool wear [min]')

# Apply feature engineering to the main feature set X.
# These features are designed to capture relationships between operational parameters.
X['torque_speed_ratio'] = X[t_col] / (X[s_col] + 1e-6)
X['wear_torque'] = X[w_col] * X[t_col]
X['wear_speed_ratio'] = X[w_col] / (X[s_col] + 1e-6)
X['torque_per_wear'] = X[t_col] / (X[w_col] + 1e-6)

# One-hot encode the 'Type' categorical column if it exists.
if 'Type' in X.columns:
    X = pd.get_dummies(X, columns=['Type'], drop_first=True)

# Split the data into training and testing sets again after feature engineering.
# test_size=0.2: 20% for testing.
# random_state=42: For reproducibility.
# stratify=y: Maintain class distribution in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize and train a RandomForestClassifier with updated hyperparameters.
# n_estimators=300: Increased number of trees for potentially better performance.
# class_weight='balanced': Addresses class imbalance by weighting classes during training.
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
)

# Fit the model on the (re-split and engineered) training data.
model.fit(X_train, y_train)
print('Model re-trained successfully with robust feature names.')

Model re-trained successfully with robust feature names.


In [ ]:
# SECTION 4: COLUMN CLEANING

import re

# Define a function to clean column names.
# This is crucial for models like XGBoost that do not support special characters
# (e.g., '[', ']', '<', '>') in column names, which can cause 'Feature shape mismatch' errors.
def clean_columns(df):
    """Cleans column names to remove characters incompatible with XGBoost (like brackets)."""
    df = df.copy() # Work on a copy to avoid modifying the original DataFrame unintentionally.
    df.columns = df.columns.astype(str) # Ensure column names are strings.
    # Replace bracket-like characters with underscores.
    df.columns = df.columns.str.replace(r'[[\]<>]', '_', regex=True)
    # Replace any non-alphanumeric characters (except underscore) with underscores.
    df.columns = df.columns.str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True)
    return df

# Apply the column cleaning function to both training and testing feature sets.
X_train = clean_columns(X_train)
X_test = clean_columns(X_test)

print("Column names sanitized for XGBoost compatibility.")

Column names sanitized for XGBoost compatibility.


In [ ]:
# --- XGBoost Model Training and Threshold Tuning ---
# Using XGBoost, a powerful gradient boosting framework, with specific strategies
# to handle class imbalance and optimize for the F2 score.

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score

# Calculate class weights for imbalance.
# 'scale_pos_weight' is a parameter in XGBoost that directly handles class imbalance
# by weighting the positive class (failure) during training. It's calculated as (count of negative examples / count of positive examples).
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos

# Define the XGBoost model with tuned hyperparameters.
# n_estimators=400: Number of boosting rounds.
# max_depth=5: Maximum depth of a tree.
# learning_rate=0.05: Step size shrinkage to prevent overfitting.
# subsample=0.9: Fraction of samples used for fitting the trees.
# colsample_bytree=0.9: Fraction of features used for fitting the trees.
# scale_pos_weight: Applied to balance classes.
# max_delta_step=1: Limits the maximum step size of each tree's weight estimation. Useful for imbalanced classes.
# eval_metric="logloss": Metric used for early stopping.
# random_state=42: For reproducibility.
# n_jobs=-1: Use all available CPU cores.
model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    max_delta_step=1,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Train the XGBoost model on the cleaned training data.
model.fit(X_train, y_train)

# Retrieve predicted probabilities for the positive class (failure) on the test set.
proba = model.predict_proba(X_test)[:, 1]

# Search for the optimal classification threshold to maximize the F2 Score.
# F2 is preferred as we prioritize Recall (correctly identifying failures) over Precision.
# np.linspace generates evenly spaced numbers over a specified interval.
thresholds = np.linspace(0.50, 0.90, 81)
results = []

# Iterate through each threshold to calculate performance metrics.
for t in thresholds:
    # Convert probabilities to binary predictions based on the current threshold.
    y_pred_t = (proba >= t).astype(int)
    # Calculate the confusion matrix and unpack its components.
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    # Calculate Precision (handling potential division by zero).
    precision = tp / (tp + fp) if (tp + fp) else 0
    # Calculate Recall (handling potential division by zero).
    recall = tp / (tp + fn) if (tp + fn) else 0
    # Calculate F2 Score (beta=2 for increased emphasis on recall).
    f2 = fbeta_score(y_test, y_pred_t, beta=2)
    # Store all metrics for the current threshold.
    results.append([t, tn, fp, fn, tp, precision, recall, f2])

# Convert results to a DataFrame for better readability and analysis.
df_results = pd.DataFrame(
    results,
    columns=["threshold", "tn", "fp", "fn", "tp", "precision", "recall", "f2"]
)

# Identify the row with the highest F2 score to determine the optimal threshold.
best_row = df_results.loc[df_results["f2"].idxmax()]
best_threshold = best_row["threshold"]

# Print the top 10 thresholds sorted by F2 score for review.
print("Top 10 Thresholds by F2 Score:")
print(df_results.sort_values("f2", ascending=False).head(10))

# Apply the optimized threshold to get the final predictions.
y_pred_final = (proba >= best_threshold).astype(int)
# Print the final evaluation metrics using the best threshold.
print(f"\nFinal Results using Threshold: {best_threshold}")
print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

Top 10 Thresholds by F2 Score:
    threshold     tn   fp   fn   tp  precision    recall        f2
63      0.815  26345  511  164  266   0.342342  0.618605  0.532639
60      0.800  26308  548  160  270   0.330073  0.627907  0.531915
58      0.790  26273  583  156  274   0.319720  0.637209  0.531626
62      0.810  26329  527  163  267   0.336272  0.620930  0.531026
71      0.855  26453  403  178  252   0.384733  0.586047  0.530526
65      0.825  26376  480  169  261   0.352227  0.606977  0.530272
78      0.890  26543  313  189  241   0.435018  0.560465  0.529903
57      0.785  26256  600  155  275   0.314286  0.639535  0.529865
59      0.795  26288  568  159  271   0.323004  0.630233  0.529504
56      0.780  26242  614  154  276   0.310112  0.641860  0.528736

Final Results using Threshold: 0.815
[[26345   511]
 [  164   266]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     26856
           1       0.34      0.62      0.44       430


In [ ]:
# SECTION 7: THRESHOLD OPTIMIZATION LOGIC
# This utility block explicitly details the process for fine-tuning the classification threshold
# to optimize the F2 score for the chosen model, which is crucial for imbalanced datasets
# where False Negatives are more costly.

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score

# Define a range of thresholds to systematically test.
# Using np.linspace for a granular search between 0.50 and 0.90.
thresholds = np.linspace(0.50, 0.90, 81)
results = [] # List to store performance metrics for each threshold.

# Loop through each threshold to evaluate its impact on model performance.
for t in thresholds:
    # Convert predicted probabilities into binary class labels based on the current threshold.
    y_pred_t = (proba >= t).astype(int)

    # Calculate the confusion matrix to get TN, FP, FN, TP values.
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    # Calculate Precision: Proportion of positive predictions that are truly positive.
    precision = tp / (tp + fp) if (tp + fp) else 0
    # Calculate Recall: Proportion of actual positives that are correctly identified.
    recall = tp / (tp + fn) if (tp + fn) else 0
    # Calculate F2 Score: A weighted harmonic mean of precision and recall,
    # giving more weight to recall (beta=2) as false negatives are more critical.
    f2 = fbeta_score(y_test, y_pred_t, beta=2)

    # Append the results for the current threshold to the list.
    results.append([t, tn, fp, fn, tp, precision, recall, f2])

# Convert the list of results into a Pandas DataFrame for easy manipulation and display.
df_results = pd.DataFrame(
    results,
    columns=["threshold", "tn", "fp", "fn", "tp", "precision", "recall", "f2"]
)

# Identify the best threshold by finding the row with the maximum F2 score.
# .idxmax() returns the index of the maximum value in the 'f2' column.
best_row = df_results.loc[df_results["f2"].idxmax()]
best_threshold = best_row["threshold"]

# Print the identified optimal threshold and its corresponding performance metrics.
print("Optimal Threshold identified:", best_threshold)
print(best_row)

Optimal Threshold identified: 0.815
threshold        0.815000
tn           26345.000000
fp             511.000000
fn             164.000000
tp             266.000000
precision        0.342342
recall           0.618605
f2               0.532639
Name: 63, dtype: float64


In [ ]:
# SECTION 5: FINAL PREDICTION FOR SUBMISSION
# This section prepares the test data, applies the trained model and optimized threshold
# to make final predictions, and generates a submission file.

import pandas as pd
import numpy as np

# Load the unseen test dataset.
test_df = pd.read_csv('/content/drive/MyDrive/Almabetter/Capstone/ML/test.csv')

# Store the 'id' column separately, as it's needed for the submission file but not for prediction.
test_ids = test_df['id'].copy()

# Prepare features from the test DataFrame, dropping 'id' and 'Product ID'.
# '.copy()' is used to avoid SettingWithCopyWarning.
X_final = test_df.drop(columns=['id', 'Product ID'], errors='ignore').copy()

# We use the original column names for feature engineering here because the model
# (specifically the RandomForest model in 'u8beCZePzHQw' or XGBoost in 'SNu7B6jx1TWB' after cleaning)
# was fitted on features generated from these original names *before* column sanitization.
t_col = 'Torque [Nm]'
s_col = 'Rotational speed [rpm]'
w_col = 'Tool wear [min]'

# Apply the same feature engineering steps to the X_final (test) dataset as was done for the training data.
X_final['torque_speed_ratio'] = X_final[t_col] / (X_final[s_col] + 1e-6)
X_final['wear_torque'] = X_final[w_col] * X_final[t_col]
X_final['wear_speed_ratio'] = X_final[w_col] / (X_final[s_col] + 1e-6)
X_final['torque_per_wear'] = X_final[t_col] / (X_final[w_col] + 1e-6)

# Handle 'Type' categorical encoding for the final test set.
# This ensures consistency with the training data's feature set.
if 'Type' in X_final.columns:
    X_final = pd.get_dummies(X_final, columns=['Type'], drop_first=True)

# Align columns of X_final with the columns of X_train (which the model was trained on).
# This is crucial to ensure the test data has the exact same features in the same order as the training data.
# fill_value=0: If any column is present in X_train but not in X_final (e.g., a 'Type' category not in test_df),
# it will be added and filled with zeros.
X_final = X_final.reindex(columns=X_train.columns, fill_value=0)

# Predict probabilities for the positive class (failure) using the trained model.
proba_test = model.predict_proba(X_final)[:, 1]

# Set the final classification threshold, which was determined to be optimal in previous steps.
final_threshold = 0.815

# Convert the probabilities into final binary predictions (0 or 1) using the optimized threshold.
pred_test = (proba_test >= final_threshold).astype(int)

# Generate the submission DataFrame with 'id' and 'Machine failure' predictions.
submission = pd.DataFrame({'id': test_ids, 'Machine failure': pred_test})

# Save the submission DataFrame to a CSV file.
# index=False prevents Pandas from writing the DataFrame index as a column in the CSV.
submission.to_csv("/content/drive/MyDrive/Almabetter/Capstone/ML/submission.csv", index=False)
print("Submission file 'submission.csv' generated successfully matching model features.")

Submission file 'submission.csv' generated successfully matching model features.
